In [ ]:
import os, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = "results"
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)


records = []
for fp in glob.glob(os.path.join(RESULTS_DIR, "metrics_*.json")):
    with open(fp, "r", encoding="utf-8") as f:
        data = json.load(f)

    rec_total = {
        "model": data["model"],
        "resolution": data["resolution"], 
        "input_width": data["input_width"],
        "horizon": data["horizon"],
        "MAE_total": data["metrics_total"]["MAE"],
        "RMSE_total": data["metrics_total"]["RMSE"],
        "path": fp,
    }
    records.append(rec_total)


df_total = pd.DataFrame(records)


step_rows = []
for fp in glob.glob(os.path.join(RESULTS_DIR, "metrics_*.json")):
    with open(fp, "r", encoding="utf-8") as f:
        d = json.load(f)
    H = d["horizon"]
    mae_steps = d["metrics_per_step"]["MAE"]
    rmse_steps = d["metrics_per_step"]["RMSE"]
    for s in range(H):
        step_rows.append({
            "model": d["model"],
            "resolution": d["resolution"],
            "input_width": d["input_width"],
            "horizon": H,
            "step": s + 1,               
            "MAE": float(mae_steps[s]),
            "RMSE": float(rmse_steps[s]),
            "path": fp,
        })
df_step = pd.DataFrame(step_rows)

def group_label(resolution, horizon):
    unit = "min" if resolution == "minute" else "h"
    return f"{horizon} {unit}"

df_total["group"] = df_total.apply(lambda r: group_label(r["resolution"], r["horizon"]), axis=1)
df_step["group"] = df_step.apply(lambda r: group_label(r["resolution"], r["horizon"]), axis=1)

print("Gefundene Ergebnisse:")
display(df_total)


In [ ]:
def plot_overall_bar(df_total, metric="MAE_total", fname=None):

    tbl = df_total.pivot_table(index="group", columns="model", values=metric, aggfunc="mean")
    ax = tbl.plot(kind="bar", figsize=(8, 5))
    ax.set_title(f"Gesamtvergleich – {metric.replace('_total','')}")
    ax.set_ylabel(metric.replace("_total", ""))
    ax.set_xlabel("Horizont")
    ax.legend(title="Modell")
    ax.grid(True, axis="y", alpha=0.3)


    if {"cnn", "ridge"}.issubset(set(tbl.columns)):
        deltas = (tbl["cnn"] - tbl["ridge"]) / tbl["ridge"] * 100.0
        for i, (grp, delta) in enumerate(deltas.items()):
            y = max(tbl.loc[grp, "cnn"], tbl.loc[grp, "ridge"])
            ax.text(i, y, f"Δ vs Ridge: {delta:+.1f}%", ha="center", va="bottom")
    plt.tight_layout()
    if fname:
        plt.savefig(os.path.join(FIG_DIR, fname), dpi=200, bbox_inches="tight")
    plt.show()

plot_overall_bar(df_total, metric="MAE_total", fname="overall_bar_MAE.png")
plot_overall_bar(df_total, metric="RMSE_total", fname="overall_bar_RMSE.png")


In [ ]:
def plot_degradation(df_step, metric="MAE", resolution=None, horizon=None, fname=None):
    sub = df_step.copy()
    if resolution is not None:
        sub = sub[sub["resolution"] == resolution]
    if horizon is not None:
        sub = sub[sub["horizon"] == horizon]

    for grp, gdf in sub.groupby("group"):
        fig, ax = plt.subplots(figsize=(8, 5))
        for model, mdf in gdf.groupby("model"):
            mdf = mdf.sort_values("step")
            ax.plot(mdf["step"], mdf[metric], marker="o", label=model)
        ax.set_title(f"Degradationskurve – {metric} – {grp}")
        ax.set_xlabel("Vorhersageschritt")
        ax.set_ylabel(metric)
        ax.grid(True, alpha=0.3)
        ax.legend(title="Modell")
        plt.tight_layout()
        out = fname.replace(".png", f"_{grp.replace(' ','_')}.png") if fname else None
        if out:
            plt.savefig(os.path.join(FIG_DIR, out), dpi=200, bbox_inches="tight")
        plt.show()

# MAE
plot_degradation(df_step, metric="MAE", fname="degradation_MAE.png")
# RMSE
plot_degradation(df_step, metric="RMSE", fname="degradation_RMSE.png")


In [ ]:
def plot_delta_curve(df_step, metric="MAE", fname=None):

    for grp, gdf in df_step.groupby("group"):
        pvt = gdf.pivot_table(index="step", columns="model", values=metric, aggfunc="mean")
        if not {"cnn", "ridge"}.issubset(set(pvt.columns)):
            continue
        delta = pvt["ridge"] - pvt["cnn"]  
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(delta.index, delta.values, marker="o")
        ax.axhline(0, linestyle="--")
        ax.set_title(f"Delta-Kurve (Ridge − CNN) – {metric} – {grp}")
        ax.set_xlabel("Vorhersageschritt")
        ax.set_ylabel(f"Δ {metric} (Ridge − CNN)")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        out = fname.replace(".png", f"_{grp.replace(' ','_')}.png") if fname else None
        if out:
            plt.savefig(os.path.join(FIG_DIR, out), dpi=200, bbox_inches="tight")
        plt.show()

plot_delta_curve(df_step, metric="MAE", fname="delta_curve_MAE.png")
plot_delta_curve(df_step, metric="RMSE", fname="delta_curve_RMSE.png")
